## Growth Comparison Across Six Models

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


PROJECT_ROOT = Path(".").resolve()

script_dir = str(PROJECT_ROOT / "script")
if script_dir not in sys.path:
    sys.path.insert(0, script_dir)

from ECMpy_function import get_enzyme_constraint_model


def set_plot_style():
    plt.rcdefaults()
    plt.rcParams.update({
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "font.family": "Arial",
        "font.size": 12,
        "axes.linewidth": 1.5,
        "xtick.major.width": 1.5,
        "ytick.major.width": 1.5,
        "xtick.minor.width": 1.0,
        "ytick.minor.width": 1.0,
        "xtick.major.size": 5,
        "ytick.major.size": 5,
        "xtick.minor.size": 3,
        "ytick.minor.size": 3,
        "xtick.direction": "out",
        "ytick.direction": "out",
    })


def format_axes(ax):
    ax.tick_params(
        axis="both", which="major",
        labelsize=11, top=False, right=False
    )
    ax.tick_params(
        axis="both", which="minor",
        top=False, right=False
    )
    ax.minorticks_on()
    ax.grid(
        True, which="major", linestyle="-",
        alpha=0.3, linewidth=1.0, color="gray"
    )
    ax.grid(
        True, which="minor", linestyle=":",
        alpha=0.2, linewidth=0.5, color="gray"
    )
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
        spine.set_edgecolor("black")


def add_legend(ax, handles=None):
    options = {
        "loc": "upper left",
        "fontsize": 10,
        "framealpha": 1.0,
        "edgecolor": "black",
        "fancybox": False,
        "facecolor": "white",
        "borderpad": 0.7,
        "labelspacing": 0.5,
        "handlelength": 2.0,
    }
    if handles is not None:
        options["handles"] = handles

    legend = ax.legend(**options)
    legend.get_frame().set_linewidth(1.5)


def resolve_project_path(path):
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def analyze_growth(model_file, model_name, uptake_range):
    model = get_enzyme_constraint_model(str(model_file))
    model.objective = model.reactions.get_by_id("Bio_cgATCC14067_c")
    model.objective_direction = "max"

    glucose = model.reactions.get_by_id("EX_cpd00027_e_reverse")
    growth_rates = np.zeros(len(uptake_range))

    for index, uptake in enumerate(uptake_range):
        glucose.bounds = (float(uptake), float(uptake))

        try:
            solution = model.optimize()
            if solution.status == "optimal":
                growth_rates[index] = solution.objective_value
            else:
                print(
                    f"{model_name}: uptake={uptake:.4f}, "
                    f"status={solution.status}"
                )
        except Exception as exc:
            print(f"{model_name}: uptake={uptake:.4f}, error={exc}")

    peak = int(np.argmax(growth_rates))
    print(
        f"{model_name}: maximum growth={growth_rates[peak]:.4f}, "
        f"uptake={uptake_range[peak]:.2f}"
    )

    return growth_rates


def compare_growth(model_files, figure_file, table_file):
    if len(model_files) != 6:
        raise ValueError("Exactly six model files are required.")

    resolved_models = {
        name: resolve_project_path(path)
        for name, path in model_files.items()
    }

    missing_files = [
        str(path) for path in resolved_models.values()
        if not path.is_file()
    ]
    if missing_files:
        raise FileNotFoundError(
            "Model files not found:\n" + "\n".join(missing_files)
        )

    figure_file = resolve_project_path(figure_file)
    table_file = resolve_project_path(table_file)

    figure_file.parent.mkdir(parents=True, exist_ok=True)
    table_file.parent.mkdir(parents=True, exist_ok=True)

    uptake_range = np.linspace(0, 40, 50)
    colors = [
        "#91CCC0", "#8B5BCC", "#7FABD1",
        "#F7AC53", "#EC6E66", "#B5CE4E"
    ]
    markers = ["o", "s", "^", "v", "D", "X"]

    results = pd.DataFrame(index=uptake_range)

    for model_name, model_file in resolved_models.items():
        results[model_name] = analyze_growth(
            model_file, model_name, uptake_range
        )

    set_plot_style()
    fig, ax = plt.subplots(figsize=(10, 8))

    for model_name, color, marker in zip(results.columns, colors, markers):
        ax.plot(
            uptake_range, results[model_name],
            color=color, linestyle="-", marker=marker,
            markevery=3, markersize=7, linewidth=2.5,
            markeredgewidth=1.2, markeredgecolor=color,
            label=model_name
        )

    ax.scatter(
        [2.33, 3.44, 4.27, 4.086, 4.8, 5.1],
        [0.2, 0.3, 0.4, 0.41, 0.42, 0.46],
        marker="o", color="black", s=100,
        edgecolor="black", linewidth=2, zorder=10,
        label="Experimental Data"
    )

    ax.set_xlim(0, 30)
    ax.set_ylim(0, 0.7)
    ax.set_xlabel(
        "Glucose Uptake Rate (mmol/gDW/h)",
        fontsize=13, fontweight="bold"
    )
    ax.set_ylabel(
        "Growth Rate (/h)",
        fontsize=13, fontweight="bold"
    )
    ax.set_title(
        "Relationship Between Substrate Uptake Rate and Growth Rate\n"
        "Comparison of Six Models",
        fontsize=14, fontweight="bold", pad=15
    )

    format_axes(ax)
    add_legend(ax)
    fig.tight_layout()

    fig.savefig(
        figure_file, dpi=600, bbox_inches="tight",
        facecolor="white", edgecolor="none", transparent=False
    )
    results.to_csv(table_file)

    print(f"Figure saved: {figure_file}")
    print(f"Table saved: {table_file}")
    plt.show()

    return results


model_files = {
    "ecCZ870_catpred": "model_Calibration/catpred_Calibration/ecCZ870_catpred_Calibration.json",
    "ecCZ870_catapro": "model_Calibration/catapro_Calibration/ecCZ870_catapro_Calibration.json",
    "ecCZ870_DLKcat": "model_Calibration/DLKcat_Calibration/ecCZ870_DLKcat_Calibration.json",
    "ecCZ870_UniKP": "model_Calibration/UniKP_Calibration/ecCZ870_UniKP_Calibration.json",
    "ecCZ870_AutoPACMEN_mean": "model_Calibration/AutoPACMEN_mean_Calibration/ecCZ870_AutoPACMEN_mean_Calibration.json",
    "ecCZ870_AutoPACMEN_median": "model_Calibration/AutoPACMEN_median_Calibration/ecCZ870_AutoPACMEN_median_Calibration.json",
}

figure_file = "comparison6model_Calibration.png"
table_file = "comparison6model_results_Calibration.csv"

results_calibration = compare_growth(
    model_files, figure_file, table_file
)

## Growth Comparison with Modified DLKcat Model

In [ ]:
model_files = {
    "ecCZ870_catpred": "model_Calibration/catpred_Calibration/ecCZ870_catpred_Calibration.json",
    "ecCZ870_catapro": "model_Calibration/catapro_Calibration/ecCZ870_catapro_Calibration.json",
    "ecCZ870_DLKcat": "model_Calibration/DLKcat_Calibration/ecCZ870_DLKcat_Calibration_modify.json",
    "ecCZ870_UniKP": "model_Calibration/UniKP_Calibration/ecCZ870_UniKP_Calibration.json",
    "ecCZ870_AutoPACMEN_mean": "model_Calibration/AutoPACMEN_mean_Calibration/ecCZ870_AutoPACMEN_mean_Calibration.json",
    "ecCZ870_AutoPACMEN_median": "model_Calibration/AutoPACMEN_median_Calibration/ecCZ870_AutoPACMEN_median_Calibration.json",
}

figure_file = "comparison6model_modify.png"
table_file = "comparison6model_results_modify.csv"

results_modified = compare_growth(
    model_files, figure_file, table_file
)

## Overflow Metabolism: iCZ870 vs. ecCZ870

In [ ]:
import cobra

def analyze_overflow(model, model_label, uptake_range):
    biomass_id = "EX_biomass_c"
    model.objective = model.reactions.get_by_id(biomass_id)

    if "EX_cpd00027_e_reverse" in model.reactions:
        glucose_id = "EX_cpd00027_e_reverse"
        reverse_exchange = True
    elif "EX_cpd00027_e" in model.reactions:
        glucose_id = "EX_cpd00027_e"
        reverse_exchange = False
    else:
        raise KeyError(f"{model_label}: glucose exchange reaction not found")

    glucose = model.reactions.get_by_id(glucose_id)
    flux_columns = {
        "growth(EX_biomass_c)": biomass_id,
        "CO2(EX_cpd00011_e)": "EX_cpd00011_e",
        "Acetate(EX_cpd00029_e)": "EX_cpd00029_e",
    }
    records = []

    for uptake in uptake_range:
        record = {
            "glc_uptake": uptake,
            "status": "not_solved",
            "glc_rxn_used": glucose_id,
            **dict.fromkeys(flux_columns, np.nan),
        }

        with model:
            glucose.bounds = (
                (float(uptake), float(uptake))
                if reverse_exchange
                else (-float(uptake), 0.0)
            )

            try:
                solution = model.optimize()
                record["status"] = solution.status

                if solution.status == "optimal":
                    for column, reaction_id in flux_columns.items():
                        record[column] = float(
                            solution.fluxes.get(reaction_id, np.nan)
                        )
            except Exception as exc:
                record["status"] = f"error:{str(exc)[:80]}"

        records.append(record)

    result = pd.DataFrame.from_records(records).set_index("glc_uptake")
    optimal_count = result["status"].eq("optimal").sum()
    print(f"{model_label}: optimal points = {optimal_count}/{len(result)}")

    return result


set_plot_style()

uptake_range = np.linspace(1, 7, 50)

model_ic = cobra.io.read_sbml_model("iCZ870_MM.xml")
model_ec = get_enzyme_constraint_model(
    "model_Calibration/DLKcat_Calibration/"
    "ecCZ870_DLKcat_Calibration_modify.json"
)

df_ic = analyze_overflow(model_ic, "iCZ870 (original)", uptake_range)
df_ec = analyze_overflow(model_ec, "ecCZ870 (DLKcat)", uptake_range)

overflow_results = pd.concat(
    [df_ic.add_prefix("iCZ870_"), df_ec.add_prefix("ecCZ870_")],
    axis=1
)

table_file = "overflow_iCZ870_vs_ecCZ870.csv"
figure_file = "overflow_iCZ870_vs_ecCZ870_combined.png"
overflow_results.to_csv(table_file)

fig, ax1 = plt.subplots(figsize=(10, 8))
ax2 = ax1.twinx()

plot_specs = (
    (ax1, df_ic, "growth(EX_biomass_c)", "iCZ870 - Growth", "#91CCC0", "o"),
    (ax1, df_ec, "growth(EX_biomass_c)", "ecCZ870 - Growth", "#F7AC53", "v"),
    (ax2, df_ic, "CO2(EX_cpd00011_e)", "iCZ870 - CO2", "#8B5BCC", "s"),
    (ax2, df_ic, "Acetate(EX_cpd00029_e)", "iCZ870 - Acetate", "#7FABD1", "^"),
    (ax2, df_ec, "CO2(EX_cpd00011_e)", "ecCZ870 - CO2", "#EC6E66", "D"),
    (ax2, df_ec, "Acetate(EX_cpd00029_e)", "ecCZ870 - Acetate", "#B5CE4E", "X"),
)

lines = []

for axis, data, column, label, color, marker in plot_specs:
    line, = axis.plot(
        data.index, data[column],
        color=color, linewidth=2.5, marker=marker,
        markevery=3, markersize=7,
        markeredgewidth=1.2, markeredgecolor=color,
        label=label
    )
    lines.append(line)

ax1.set_xlabel(
    "Glucose Uptake Rate (mmol/gDW/h)",
    fontsize=13, fontweight="bold"
)
ax1.set_ylabel("Growth Rate (/h)", fontsize=13, fontweight="bold")
ax2.set_ylabel(
    "CO2 & Acetate Flux (mmol/gDW/h)",
    fontsize=13, fontweight="bold"
)
ax1.set_title(
    "Overflow Metabolism with Increasing Glucose Uptake\n"
    "Comparison of iCZ870 and ecCZ870 Models",
    fontsize=14, fontweight="bold", pad=15
)
ax1.set_ylim(0, 1)

format_axes(ax1)
ax2.tick_params(
    axis="y", which="major", labelsize=11,
    left=False, right=True, labelleft=False, labelright=True,
    width=1.5, labelcolor="black"
)
ax2.tick_params(
    axis="y", which="minor", left=False, right=True, width=1.0
)
for spine in ax2.spines.values():
    spine.set_linewidth(1.5)
    spine.set_edgecolor("black")

add_legend(ax1, handles=lines)
fig.tight_layout()
fig.savefig(
    figure_file, dpi=600, bbox_inches="tight",
    facecolor="white", edgecolor="none", transparent=False
)

print(f"Figure saved: {figure_file}")
print(f"Table saved: {table_file}")
plt.show()